# Product Return Prediction – Machine Learning Project

Beginner-friendly binary classification project.

**Target:** `Returned` — 1 = Product Returned, 0 = Product Not Returned.

This notebook covers data loading, cleaning, EDA, feature engineering, preprocessing, Logistic Regression, Random Forest, evaluation, feature importance, prediction, and model saving.

**Prediction timing:** This version assumes prediction happens after delivery, so actual `Delivery_Days` is available. `Return_Reason` is never used as a feature because it is generally known only after a return.

## 1. Import libraries

In [ ]:
import json, warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Dataset path

Place your real CSV at `dataset/ecommerce_returns.csv`. If it is not present, the notebook creates a reproducible 7,000-row synthetic dataset. Synthetic results demonstrate the workflow only; they do not establish real-world predictive accuracy.

In [ ]:
DATA_PATH = Path("dataset/ecommerce_returns.csv")
OUTPUT_DIR = Path("models")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load or generate data

Expected columns: `Order_ID`, `Product_Category`, `Product_Price`, `Quantity`, `Discount_Percentage`, `Delivery_Days`, `Payment_Method`, `Customer_Order_Count`, `Previous_Return_Count`, `Product_Rating`, `Shipping_Type`, and `Returned`.

In [ ]:
def create_synthetic_dataset(n_rows=7000, random_state=42):
    rng = np.random.default_rng(random_state)
    categories = ["Electronics", "Fashion", "Shoes", "Grocery", "Books"]
    category = rng.choice(categories, n_rows, p=[.22, .28, .20, .18, .12])
    ranges = {"Electronics": (800,30000), "Fashion": (300,5000),
              "Shoes": (700,7000), "Grocery": (50,1500), "Books": (100,2000)}
    price = np.array([rng.uniform(*ranges[c]) for c in category]).round(2)
    qty = rng.integers(1, 5, n_rows)
    discount = rng.uniform(0, 50, n_rows).round(1)
    delivery = np.clip(rng.poisson(4, n_rows)+1, 1, 14)
    payment = rng.choice(["COD","UPI","Card","Net Banking"], n_rows, p=[.30,.30,.30,.10])
    orders = rng.integers(0, 31, n_rows)
    previous_returns = np.array([rng.integers(0, x+1) if x else 0 for x in orders])
    rating = np.clip(rng.normal(3.8,.8,n_rows),1,5).round(1)
    shipping = rng.choice(["Standard","Express"], n_rows, p=[.72,.28])
    cat_effect = pd.Series(category).map({"Electronics":.25,"Fashion":.65,"Shoes":.55,
                                         "Grocery":-.55,"Books":-.25}).to_numpy()
    history_rate = previous_returns / np.maximum(orders, 1)
    # Labels intentionally depend on input features (synthetic teaching data).
    logit = (-2 + cat_effect + .055*(delivery-4) + .018*(discount-15)
             + 1.8*history_rate + .28*(3.8-rating)
             + .12*(payment=="COD") + .08*(shipping=="Standard") + .06*(qty-1))
    p = 1/(1+np.exp(-logit))
    returned = rng.binomial(1,p)
    return pd.DataFrame({
        "Order_ID":[f"ORD{i:05d}" for i in range(1,n_rows+1)],
        "Product_Category":category, "Product_Price":price, "Quantity":qty,
        "Discount_Percentage":discount, "Delivery_Days":delivery,
        "Payment_Method":payment, "Customer_Order_Count":orders,
        "Previous_Return_Count":previous_returns, "Product_Rating":rating,
        "Shipping_Type":shipping, "Returned":returned
    })

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    DATA_SOURCE = "User-provided CSV"
else:
    df = create_synthetic_dataset()
    df.to_csv(DATA_PATH, index=False)
    DATA_SOURCE = "Synthetic data generated by notebook"

print(DATA_SOURCE, df.shape)
display(df.head())

## 4. Understand and clean the data

In [ ]:
display(df.info())
display(df.describe(include="all").T)
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

required = ["Order_ID","Product_Category","Product_Price","Quantity",
"Discount_Percentage","Delivery_Days","Payment_Method","Customer_Order_Count",
"Previous_Return_Count","Product_Rating","Shipping_Type","Returned"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.drop_duplicates().drop_duplicates(subset=["Order_ID"]).copy()
numeric = ["Product_Price","Quantity","Discount_Percentage","Delivery_Days",
"Customer_Order_Count","Previous_Return_Count","Product_Rating","Returned"]
for col in numeric:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["Returned"])
df = df[df["Returned"].isin([0,1])]
rules = {
    "Product_Price": df["Product_Price"] >= 0,
    "Quantity": df["Quantity"] > 0,
    "Discount_Percentage": df["Discount_Percentage"].between(0,100),
    "Delivery_Days": df["Delivery_Days"] >= 0,
    "Customer_Order_Count": df["Customer_Order_Count"] >= 0,
    "Previous_Return_Count": df["Previous_Return_Count"] >= 0,
    "Product_Rating": df["Product_Rating"].between(1,5)
}
for col, mask in rules.items():
    df.loc[~mask, col] = np.nan
df.loc[df["Previous_Return_Count"] > df["Customer_Order_Count"],
       "Previous_Return_Count"] = np.nan
for col in ["Product_Category","Payment_Method","Shipping_Type"]:
    df[col] = df[col].astype("string").str.strip().replace("", pd.NA)
df["Returned"] = df["Returned"].astype(int)
print("Cleaned shape:", df.shape)
display(df.head())

## 5. Exploratory Data Analysis (EDA)

In [ ]:
print("Overall return rate:", f"{df.Returned.mean():.2%}")
display(df["Returned"].value_counts().rename(index={0:"Not Returned",1:"Returned"}))
sns.countplot(data=df, x="Returned")
plt.title("Returned vs Not Returned")
plt.xticks([0,1], ["Not Returned","Returned"])
plt.show()

display(df.groupby("Product_Category")["Returned"].agg(["mean","count"]).sort_values("mean",ascending=False))
sns.barplot(data=df, x="Product_Category", y="Returned", errorbar=None)
plt.title("Return rate by product category")
plt.xticks(rotation=25)
plt.show()

sns.barplot(data=df, x="Delivery_Days", y="Returned", errorbar=None)
plt.title("Return rate by delivery days")
plt.show()

df["Discount_Group_EDA"] = pd.cut(df["Discount_Percentage"],
    [-.01,10,30,100], labels=["Low (0-10%)","Medium (10-30%)","High (above 30%)"])
display(df.groupby("Discount_Group_EDA", observed=False)["Returned"].agg(["mean","count"]))
sns.barplot(data=df, x="Discount_Group_EDA", y="Returned", errorbar=None)
plt.title("Return rate by discount group")
plt.show()

sns.barplot(data=df, x="Previous_Return_Count", y="Returned", errorbar=None)
plt.title("Current return rate by previous return count")
plt.show()

## 6. Feature engineering

Create customer return rate, discount category, and customer type from historical information only. Never use the current order's eventual return label to calculate customer history.

In [ ]:
df["Customer_Return_Rate"] = (
    df["Previous_Return_Count"] / df["Customer_Order_Count"].replace(0,np.nan)
).fillna(0)
df["Discount_Category"] = pd.cut(df["Discount_Percentage"],
    [-.01,10,30,100], labels=["Low","Medium","High"]).astype("object")
df["Customer_Type"] = pd.cut(df["Customer_Order_Count"],
    [-1,0,5,np.inf],
    labels=["New Customer","Regular Customer","Frequent Customer"]).astype("object")
display(df[["Customer_Order_Count","Previous_Return_Count","Customer_Return_Rate",
            "Discount_Percentage","Discount_Category","Customer_Type","Returned"]].head())

## 7. Prepare features and target

`Order_ID` is only an identifier. `Return_Reason`, refund status, and other post-return information must not be used as model inputs.

In [ ]:
feature_columns = ["Product_Category","Product_Price","Quantity",
"Discount_Percentage","Delivery_Days","Payment_Method","Customer_Order_Count",
"Previous_Return_Count","Product_Rating","Shipping_Type","Customer_Return_Rate",
"Discount_Category","Customer_Type"]
categorical_features = ["Product_Category","Payment_Method","Shipping_Type",
                        "Discount_Category","Customer_Type"]
numerical_features = [c for c in feature_columns if c not in categorical_features]
X = df[feature_columns].copy()
y = df["Returned"].copy()
display(y.value_counts(normalize=True).rename("Class proportion"))

## 8. Train-test split

Stratification keeps the class proportions approximately consistent.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print("Train return rate:", y_train.mean(), "Test return rate:", y_test.mean())

## 9. Preprocessing pipeline

Numerical columns use median imputation and standard scaling. Categorical columns use most-frequent imputation and one-hot encoding. Preprocessing is fitted only on training data.

In [ ]:
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([("imputer",SimpleImputer(strategy="median")),
                         ("scaler",StandardScaler())])
categorical_pipe = Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                             ("onehot",encoder)])
preprocessor = ColumnTransformer([
    ("num",numeric_pipe,numerical_features),
    ("cat",categorical_pipe,categorical_features)
])

## 10. Train Logistic Regression and Random Forest

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor",preprocessor),
        ("classifier",LogisticRegression(max_iter=2000,class_weight="balanced",
                                         random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor",preprocessor),
        ("classifier",RandomForestClassifier(n_estimators=250,max_depth=15,
            min_samples_leaf=2,class_weight="balanced",random_state=42,n_jobs=-1))
    ])
}
for name, model in models.items():
    print("Training", name)
    model.fit(X_train,y_train)
print("Training complete.")

## 11. Model evaluation

Compare accuracy, precision, recall, F1-score, and ROC-AUC. Recall matters for identifying likely returns; precision matters because false alarms also create operational costs.

In [ ]:
rows, predictions = [], {}
for name, model in models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]
    predictions[name] = (pred,prob)
    rows.append({"Model":name,
        "Accuracy":accuracy_score(y_test,pred),
        "Precision":precision_score(y_test,pred,zero_division=0),
        "Recall":recall_score(y_test,pred,zero_division=0),
        "F1_Score":f1_score(y_test,pred,zero_division=0),
        "ROC_AUC":roc_auc_score(y_test,prob)})
results_df = pd.DataFrame(rows).set_index("Model").sort_values("F1_Score",ascending=False)
display(results_df.style.format("{:.3f}"))
for name, (pred,prob) in predictions.items():
    cm = confusion_matrix(y_test,pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Not Returned","Returned"],
        yticklabels=["Not Returned","Returned"])
    plt.title(f"Confusion Matrix – {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
    print(name)
    print(classification_report(y_test,pred,
        target_names=["Not Returned","Returned"],zero_division=0))

## 12. Select and save the best model

The notebook uses the highest test F1-score as a simple baseline selection rule. A production threshold should be selected using validation data and business costs.

In [ ]:
best_model_name = results_df.index[0]
best_model = models[best_model_name]
print("Selected:", best_model_name)
model_path = OUTPUT_DIR / "return_prediction_model.joblib"
joblib.dump(best_model, model_path)

metadata = {
    "project":"Product Return Prediction",
    "selected_model":best_model_name,
    "target":"Returned",
    "features":feature_columns,
    "threshold":0.50,
    "prediction_timing":"After delivery; actual Delivery_Days is available",
    "data_source":DATA_SOURCE
}
with open(OUTPUT_DIR / "model_metadata.json","w",encoding="utf-8") as f:
    json.dump(metadata,f,indent=4)
results_df.to_csv(OUTPUT_DIR / "model_evaluation.csv")
print("Saved:", model_path)

## 13. Feature importance

Permutation importance estimates how much F1-score falls when one feature is shuffled. It is useful for interpretation but does not prove causation.

In [ ]:
perm = permutation_importance(best_model,X_test,y_test,n_repeats=5,
                              random_state=42,scoring="f1",n_jobs=-1)
importance_df = pd.DataFrame({
    "Feature":feature_columns,
    "Importance_Mean":perm.importances_mean,
    "Importance_Std":perm.importances_std
}).sort_values("Importance_Mean",ascending=False)
display(importance_df)
sns.barplot(data=importance_df,x="Importance_Mean",y="Feature")
plt.title("Permutation Feature Importance")
plt.show()

## 14. Predict a new order

Change the sample values and run the cell. The probability is a model estimate, not a guarantee.

In [ ]:
sample_order = pd.DataFrame([{
    "Product_Category":"Fashion",
    "Product_Price":2500.0,
    "Quantity":1,
    "Discount_Percentage":25.0,
    "Delivery_Days":7,
    "Payment_Method":"UPI",
    "Customer_Order_Count":12,
    "Previous_Return_Count":4,
    "Product_Rating":3.5,
    "Shipping_Type":"Standard",
    "Customer_Return_Rate":4/12,
    "Discount_Category":"Medium",
    "Customer_Type":"Frequent Customer"
}])
probability = best_model.predict_proba(sample_order)[0,1]
prediction = int(probability >= .50)
print(f"Return probability: {probability:.2%}")
print("Prediction:", "Returned" if prediction else "Not Returned")

## 15. Bulk prediction template

Upload a new CSV with the same feature columns to predict multiple orders.

In [ ]:
def predict_orders(input_df, trained_model, threshold=0.50):
    missing = [c for c in feature_columns if c not in input_df.columns]
    if missing:
        raise ValueError(f"Missing input columns: {missing}")
    output = input_df.copy()
    probabilities = trained_model.predict_proba(output[feature_columns])[:,1]
    output["Return_Probability"] = probabilities
    output["Predicted_Returned"] = (probabilities >= threshold).astype(int)
    output["Prediction_Label"] = output["Predicted_Returned"].map(
        {0:"Not Returned",1:"Returned"})
    return output

# new_orders = pd.read_csv("dataset/new_orders.csv")
# bulk_results = predict_orders(new_orders, best_model)
# bulk_results.to_csv("models/bulk_return_predictions.csv", index=False)
# display(bulk_results.head())

## 16. Load the saved model

In [ ]:
loaded_model = joblib.load(model_path)
loaded_probability = loaded_model.predict_proba(sample_order)[0,1]
print(f"Loaded model return probability: {loaded_probability:.2%}")

## 17. Business interpretation and limitations

- The model can help estimate which delivered orders may be returned and support return-handling and inventory planning.
- A high probability does not mean a customer will definitely return the product.
- Monitor both precision and recall; do not judge the model using accuracy alone.
- Synthetic labels are generated from programmed relationships. Performance on this data does not establish real-world predictive accuracy.
- For prediction before delivery, replace actual `Delivery_Days` with estimated delivery time and retrain.
- Never use `Return_Reason`, refund status, or information generated after a return as model features.
- Build the Streamlit dashboard after completing and validating this notebook.

## 18. Generated project files

After running all cells, the `models/` folder contains:

- `return_prediction_model.joblib` – selected trained pipeline
- `model_metadata.json` – model and feature details
- `model_evaluation.csv` – comparison metrics

**Next step:** Build a Streamlit application with a dashboard, single-order prediction, business insights, and bulk CSV prediction.